# EDA Completo - Telecom Churn (Base Extended)

Este notebook foi reconstruido do zero com foco em clareza e tomada de decisao.

Pergunta central: **a base extended esta pronta para os baselines da Fase 1?**

Roteiro:
1. Carregamento e contexto da base
2. Qualidade estrutural dos dados
3. Distribuicao do target e sinais de churn
4. Data readiness e risco de leakage
5. Conclusao executiva

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)

DATA_PATH = Path("../data/raw/telecom_churn_base_extended.csv")
TARGET_COL = "churn"
ID_COL = "customer_id"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Arquivo nao encontrado: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)

date_cols = ["contract_renewal_date", "loyalty_end_date"]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

numeric_cols = df.select_dtypes(include="number").columns.tolist()
feature_numeric_cols = [col for col in numeric_cols if col != TARGET_COL]
categorical_cols = df.select_dtypes(exclude="number").columns.tolist()

print(f"Arquivo: {DATA_PATH}")
print(f"Linhas: {len(df):,} | Colunas: {df.shape[1]}")
print(f"Churn rate: {df[TARGET_COL].mean():.2%}")
display(df.head(3))

## 1) Entendimento da base

Nesta etapa verificamos:
- volumetria e estrutura
- tipos de variaveis
- consistencia de ID e target
- mapa geral de blocos de negocio

In [ ]:
overview = pd.DataFrame(
    {
        "metrica": [
            "linhas",
            "colunas",
            "memoria_mb",
            "id_unico",
            "target_binario",
            "n_colunas_numericas",
            "n_colunas_categoricas",
        ],
        "valor": [
            len(df),
            df.shape[1],
            round(df.memory_usage(deep=True).sum() / 1024**2, 2),
            bool(df[ID_COL].is_unique),
            sorted(df[TARGET_COL].dropna().unique().tolist()) == [0, 1],
            len(numeric_cols),
            len(categorical_cols),
        ],
    }
)
display(overview)

schema = (
    pd.DataFrame(
        {
            "coluna": df.columns,
            "dtype": df.dtypes.astype(str).values,
            "missing": df.isna().sum().values,
            "missing_pct": (df.isna().mean() * 100).round(2).values,
            "n_unique": df.nunique(dropna=True).values,
            "unique_ratio": (df.nunique(dropna=True) / len(df)).round(4).values,
        }
    )
    .sort_values(["missing_pct", "unique_ratio"], ascending=[False, False])
    .reset_index(drop=True)
)
display(schema.head(25))

blocos = {
    "perfil_contrato": ["age", "gender", "region", "plan_type", "months_to_contract_end", "has_loyalty"],
    "qualidade_servico": ["network_outages_30d", "avg_signal_quality", "call_drop_rate", "service_failures_30d"],
    "uso_engajamento": ["minutes_monthly", "data_gb_monthly", "usage_delta_pct", "app_login_30d"],
    "financeiro": ["monthly_charges", "invoice_shock_flag", "late_payments_6m", "default_flag"],
    "atendimento": ["support_calls_30d", "complaints_30d", "resolution_time_avg"],
    "satisfacao": ["nps_score", "nps_category", "csat_score"],
}

blocos_resumo = pd.DataFrame(
    [
        {
            "bloco": nome,
            "qtd_colunas_presentes": len([c for c in cols if c in df.columns]),
            "colunas": ", ".join([c for c in cols if c in df.columns]),
        }
        for nome, cols in blocos.items()
    ]
)
display(blocos_resumo)

## 2) Qualidade dos dados

Objetivo desta secao: checar se a base e confiavel para baseline.

Vamos avaliar:
- duplicidade de linhas e IDs
- missing e sua distribuicao
- variaveis com possivel comportamento extremo

In [ ]:
quality_flags = pd.DataFrame(
    {
        "check": [
            "IDs duplicados",
            "Linhas duplicadas",
            "Colunas constantes",
            "Colunas com missing",
            "Colunas com alta cardinalidade (>30%)",
        ],
        "resultado": [
            int(df[ID_COL].duplicated().sum()),
            int(df.duplicated().sum()),
            int((df.nunique(dropna=False) <= 1).sum()),
            int((df.isna().sum() > 0).sum()),
            int(((df.nunique(dropna=True) / len(df)) > 0.3).sum()),
        ],
    }
)
display(quality_flags)

missing_summary = (
    pd.DataFrame(
        {
            "coluna": df.columns,
            "missing": df.isna().sum().values,
            "missing_pct": (df.isna().mean() * 100).round(2).values,
        }
    )
    .query("missing > 0")
    .sort_values("missing_pct", ascending=False)
    .reset_index(drop=True)
)
display(missing_summary if not missing_summary.empty else pd.DataFrame({"status": ["Sem missing"]}))

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
if missing_summary.empty:
    axes[0].text(0.5, 0.5, "Sem valores ausentes", ha="center", va="center", fontsize=12)
    axes[0].set_axis_off()
else:
    sns.barplot(data=missing_summary, x="missing_pct", y="coluna", ax=axes[0], color="#4472c4")
    axes[0].set_title("Missing por coluna")
    axes[0].set_xlabel("Missing (%)")
    axes[0].set_ylabel("")

sns.heatmap(df.isna(), cbar=False, yticklabels=False, ax=axes[1], cmap="viridis")
axes[1].set_title("Mapa de missing")
axes[1].set_xlabel("Colunas")
axes[1].set_ylabel("Linhas")
plt.tight_layout()
plt.show()

outlier_cols = ["monthly_charges", "data_gb_monthly", "support_calls_90d", "days_past_due"]
outlier_cols = [c for c in outlier_cols if c in df.columns]

if outlier_cols:
    fig, axes = plt.subplots(1, len(outlier_cols), figsize=(5 * len(outlier_cols), 4))
    if len(outlier_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, outlier_cols):
        sns.boxplot(data=df, y=col, ax=ax, color="#8ecae6")
        ax.set_title(f"Boxplot - {col}")
    plt.tight_layout()
    plt.show()

## 3) Distribuicao do target e sinais de churn

Aqui olhamos se o target esta balanceado e quais grupos mostram risco relativo maior.

Nao e teste causal: e um diagnostico para orientar os baselines.

In [ ]:
target_summary = (
    df[TARGET_COL]
    .value_counts(normalize=True)
    .rename_axis(TARGET_COL)
    .reset_index(name="pct")
)
target_summary["pct"] = target_summary["pct"] * 100
display(target_summary)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sns.barplot(data=target_summary, x=TARGET_COL, y="pct", ax=axes[0, 0], color="#2a9d8f")
axes[0, 0].set_title("Distribuicao do target")
axes[0, 0].set_xlabel("Churn")
axes[0, 0].set_ylabel("Percentual (%)")

plan_churn = (
    df.groupby("plan_type", observed=False)[TARGET_COL]
    .agg(churn_rate="mean", clientes="size")
    .sort_values("churn_rate", ascending=False)
    .reset_index()
)
plan_churn["churn_rate"] = plan_churn["churn_rate"] * 100
sns.barplot(data=plan_churn, x="plan_type", y="churn_rate", ax=axes[0, 1], color="#e76f51")
axes[0, 1].set_title("Churn rate por tipo de plano")
axes[0, 1].set_xlabel("Plano")
axes[0, 1].set_ylabel("Churn rate (%)")

nps_churn = (
    df.groupby("nps_category", observed=False)[TARGET_COL]
    .agg(churn_rate="mean", clientes="size")
    .sort_values("churn_rate", ascending=False)
    .reset_index()
)
nps_churn["churn_rate"] = nps_churn["churn_rate"] * 100
sns.barplot(data=nps_churn, x="nps_category", y="churn_rate", ax=axes[1, 0], color="#f4a261")
axes[1, 0].set_title("Churn rate por NPS")
axes[1, 0].set_xlabel("Categoria NPS")
axes[1, 0].set_ylabel("Churn rate (%)")

default_churn = (
    df.groupby("default_flag", observed=False)[TARGET_COL]
    .agg(churn_rate="mean", clientes="size")
    .sort_values("churn_rate", ascending=False)
    .reset_index()
)
default_churn["churn_rate"] = default_churn["churn_rate"] * 100
sns.barplot(data=default_churn, x="default_flag", y="churn_rate", ax=axes[1, 1], color="#457b9d")
axes[1, 1].set_title("Churn rate por inadimplencia")
axes[1, 1].set_xlabel("Default flag")
axes[1, 1].set_ylabel("Churn rate (%)")

plt.tight_layout()
plt.show()

display(plan_churn)
display(nps_churn)
display(default_churn)

In [ ]:
selected_numeric = [
    "tenure_months",
    "monthly_charges",
    "usage_delta_pct",
    "support_calls_90d",
    "nps_score",
]
selected_numeric = [c for c in selected_numeric if c in df.columns]

fig, axes = plt.subplots(len(selected_numeric), 2, figsize=(14, 4 * len(selected_numeric)))
if len(selected_numeric) == 1:
    axes = [axes]

for idx, col in enumerate(selected_numeric):
    sns.histplot(data=df, x=col, kde=True, ax=axes[idx][0], color="#264653")
    axes[idx][0].set_title(f"Distribuicao - {col}")

    sns.boxplot(data=df, x=TARGET_COL, y=col, ax=axes[idx][1], color="#8ecae6")
    axes[idx][1].set_title(f"{col} por churn")
    axes[idx][1].set_xlabel("Churn")

plt.tight_layout()
plt.show()

corr_target = (
    df[feature_numeric_cols + [TARGET_COL]]
    .corr(numeric_only=True)[[TARGET_COL]]
    .sort_values(TARGET_COL, ascending=False)
    .reset_index()
    .rename(columns={"index": "feature", TARGET_COL: "corr_com_churn"})
)

display(corr_target.head(15))
display(corr_target.tail(15))

## 4) Data readiness e risco de leakage

Nesta secao, traduzimos EDA em regra de modelagem.

Perguntas chave:
- quais colunas podem entrar no baseline
- quais colunas devem ser removidas por leakage
- quais exigem governanca temporal

In [ ]:
window_sensitive_cols = {
    "usage_delta_pct",
    "days_past_due",
    "support_calls_30d",
    "support_calls_90d",
    "complaints_30d",
    "tickets_open_30d",
    "days_since_last_usage",
    "days_since_last_interaction",
}

potential_leakage_cols = {
    "churn_probability",
    "retention_offer_accepted",
    "retention_offer_made",
}

readiness = pd.DataFrame(
    {
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "missing_pct": (df.isna().mean() * 100).round(2).values,
        "n_unique": df.nunique(dropna=True).values,
        "unique_ratio": (df.nunique(dropna=True) / len(df)).round(4).values,
    }
)
readiness["role"] = readiness["column"].map(
    lambda c: "id" if c == ID_COL else "target" if c == TARGET_COL else "feature"
)
readiness["constant"] = readiness["column"].map(lambda c: df[c].nunique(dropna=False) <= 1)
readiness["high_cardinality"] = readiness["unique_ratio"] > 0.3
readiness["window_sensitive"] = readiness["column"].isin(window_sensitive_cols)
readiness["potential_leakage"] = readiness["column"].isin(potential_leakage_cols)
readiness["recomendacao"] = readiness["column"].map(
    lambda c: (
        "remover do treino"
        if c == ID_COL
        else "excluir de features (sinal de leakage)"
        if c in potential_leakage_cols
        else "manter com governanca de janela temporal"
        if c in window_sensitive_cols
        else "ok para baseline"
    )
)

display(
    readiness.sort_values(
        ["potential_leakage", "window_sensitive", "missing_pct", "unique_ratio"],
        ascending=[False, False, False, False],
    )
)

readiness_checks = pd.DataFrame(
    [
        ["Volume para baseline", "OK", f"Base com {len(df):,} linhas e {df.shape[1]} colunas."],
        ["Qualidade estrutural", "OK", "Sem duplicidade de linha; customer_id unico."],
        ["Missing", "ATENCAO", "Missing relevante em loyalty_end_date e residual em gender."],
        ["Balanceamento do target", "OK", f"Churn em {df[TARGET_COL].mean():.2%}; adequado para baseline."],
        ["Leakage potencial", "ATENCAO", "Excluir churn_probability e colunas de retencao do treino."],
        ["Governanca temporal", "ATENCAO", "Definir janela de observacao para variaveis operacionais recentes."],
        ["Pronto para modelagem", "PARCIAL", "Pronto para baseline com split estratificado e lista de exclusao."],
    ],
    columns=["check", "status", "observacao"],
)
display(readiness_checks)

## 5) Conclusao executiva

Diagnostico final:
- A base extended esta pronta para iniciar baselines supervisionados.
- O volume e a estrutura sao suficientes para a Fase 1.
- O principal risco e leakage por colunas de retencao e variaveis proximas do evento de churn.

Acoes obrigatorias para o proximo notebook (baselines):
1. Excluir `customer_id`, `churn_probability`, `retention_offer_made` e `retention_offer_accepted`.
2. Definir preprocessamento separado para numericas e categoricas.
3. Validar com split estratificado e metricas AUC-ROC, PR-AUC e Recall@Top-K.
4. Registrar no MLflow a versao da base e a lista final de features.